In [ ]:
import requests
import json
import pandas as pd
from tqdm.auto import tqdm
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- CONFIGURATION ---
OPENROUTER_API_KEY
MODEL_NAME = "x-ai/grok-4.1-fast"
INPUT_FILE = "test_data_subtask_1.json"
OUTPUT_FILE = "predictions_fewshot.json"
MAX_WORKERS = 20

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {OPENROUTER_API_KEY}"})

# --- 1. THE 46% LOGIC (Restored) ---
VALID_FORMS = {
    "AAA-1", "EAE-1", "AII-1", "EIO-1", "AAI-1", "EAO-1",
    "EAE-2", "AEE-2", "EIO-2", "AOO-2", "AEO-2", "EAO-2",
    "IAI-3", "AII-3", "OAO-3", "EIO-3", "AAI-3", "EAO-3",
    "AEE-4", "IAI-4", "EIO-4", "AAI-4", "AEO-4", "EAO-4"
}

def get_mood(s):
    # THE EXACT LOGIC FROM YOUR 46% RUN
    s = s.lower().strip()
    if s.startswith("all"): return "A"
    if s.startswith("no"): return "E"
    if s.startswith("some") and "not" in s: return "O"
    if s.startswith("some"): return "I"
    return None

def get_vars(text):
    # Matches "All A are B" -> ['A', 'B']
    words = text.split()
    # Simple heuristic: The single capital letters are variables
    vars_found = [w for w in words if len(w) == 1 and w.isupper()]
    return vars_found

def check_logic(data):
    # THE EXACT LOGIC FROM YOUR 46% RUN (Simplified for stability)
    try:
        p1 = data.get("p1_mapped", "").strip()
        p2 = data.get("p2_mapped", "").strip()
        conc = data.get("conc_mapped", "").strip()

        m1 = get_mood(p1)
        m2 = get_mood(p2)
        mc = get_mood(conc)

        if not all([m1, m2, mc]): return False

        v1 = get_vars(p1)
        v2 = get_vars(p2)
        vc = get_vars(conc)

        if len(v1)!=2 or len(v2)!=2 or len(vc)!=2: return False

        S, P = vc[0], vc[1]
        all_vars = v1 + v2
        M_list = [x for x in all_vars if x != S and x != P]
        if not M_list: return False
        M = M_list[0]

        # Determine Figure & Major/Minor (Restored Logic)
        major_mood, minor_mood = None, None
        if P in v1:
            major_mood, major_vars = m1, v1
            minor_mood, minor_vars = m2, v2
        elif P in v2:
            major_mood, major_vars = m2, v2
            minor_mood, minor_vars = m1, v1
        else: return False

        if major_vars[0] == M and minor_vars[1] == M: fig = "1"
        elif major_vars[1] == M and minor_vars[1] == M: fig = "2"
        elif major_vars[0] == M and minor_vars[0] == M: fig = "3"
        elif major_vars[1] == M and minor_vars[0] == M: fig = "4"
        else: return False

        key = f"{major_mood}{minor_mood}{mc}-{fig}"
        return key in VALID_FORMS

    except:
        return False

# --- 2. THE FEW-SHOT TRANSLATOR (The Upgrade) ---
# We teach the LLM how to normalize messy phrasing specifically.
TRANSLATOR_PROMPT = """Task: Translate Syllogism to Standard Variables (A, B, C).

MAP:
- S (Subject of Conclusion) = **A**
- P (Predicate of Conclusion) = **C**
- M (Middle Term) = **B**

STANDARD FORMS:
- All A are B
- No A are B
- Some A are B
- Some A are not B

EXAMPLES (Study these messy inputs):
1. Input: "Not a single bird is a cat." -> Output: "No A are C"
2. Input: "There exist some dogs that bark." -> Output: "Some A are B"
3. Input: "It is not the case that all tigers are lions." -> Output: "Some A are not C"
4. Input: "Every single car is a vehicle." -> Output: "All A are C"
5. Input: "None of the men are tall." -> Output: "No A are B"

INPUT TEXT: "{text}"

INSTRUCTIONS:
- Identify terms S, P, M.
- Map strictly to A, B, C.
- Rewrite into STANDARD FORM using A, B, C.
- Handle "Not a single" as "No". Handle "Not all" as "Some not".

OUTPUT JSON:
{{
  "p1_mapped": "...",
  "p2_mapped": "...",
  "conc_mapped": "..."
}}
"""

def solve_row(row):
    text = row['syllogism']
    clean_text = text.replace('"', '').strip()

    for _ in range(3):
        try:
            resp = session.post(
                "https://openrouter.ai/api/v1/chat/completions",
                json={
                    "model": MODEL_NAME,
                    "messages": [{"role": "user", "content": TRANSLATOR_PROMPT.format(text=clean_text)}],
                    "temperature": 0.0,
                    "response_format": {"type": "json_object"}
                },
                timeout=15
            )
            if resp.status_code == 200:
                data = json.loads(resp.json()['choices'][0]['message']['content'])
                is_valid = check_logic(data)
                return {"id": str(row['id']), "validity": bool(is_valid)}
            elif resp.status_code == 429: time.sleep(2)
            else: time.sleep(1)
        except: time.sleep(1)

    return {"id": str(row['id']), "validity": False}

def run_pipeline():
    try: df = pd.DataFrame(json.load(open(INPUT_FILE)))
    except: return

    print(f"🚀 Running Few-Shot Restored (46% Base + Better Prompt)...")
    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(solve_row, row): row for _, row in df.iterrows()}
        for future in tqdm(as_completed(futures), total=len(df)):
            results.append(future.result())

    results.sort(key=lambda x: x['id'])

    with open(OUTPUT_FILE, 'w') as f: json.dump(results, f, indent=4)

    import zipfile
    with zipfile.ZipFile("predictions_fewshot.zip", 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(OUTPUT_FILE, arcname="predictions.json")

    print("✅ Ready: predictions_fewshot.zip")

if __name__ == "__main__":
    run_pipeline()

🚀 Running Few-Shot Restored (46% Base + Better Prompt)...


  0%|          | 0/191 [00:00<?, ?it/s]

✅ Ready: predictions_fewshot.zip
